# 10 — Thesis tables and figures

Select completed real-data bundles below. This notebook reads aggregate results only;
it never refits a model. Original H3 results without adequate numerical verification
remain marked for rerun. Keep conditional confidence intervals and diagnostic limitations
visible. After a new run, update the paths and review the German draft before use.

In [ ]:
from pathlib import Path
from html.parser import HTMLParser
import hashlib
import json
import sys

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import numpy as np
import pandas as pd

PROJECT_DIR = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                   if (p / "ANAL").is_dir())
sys.path.insert(0, str(PROJECT_DIR))
from FIG.plotting.apa7 import set_apa7_style, apa7_axes
set_apa7_style()

# Select completed bundles explicitly. After a rerun, replace these three paths.
RESULT_BUNDLES = {
    size: PROJECT_DIR / "AIRGAPPED_RESULTS" / "20261027_real_runs" / f"{size}m" / "bundle"
    for size in [100,200,500]
}
OUTPUT_DIR = PROJECT_DIR / "ANAL" / "data" / "thesis_results" / "20261027_review"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Read and compare the selected runs

In [ ]:
class DisplayTable(HTMLParser):
    """Recover a saved pandas display; it contains rounded, not full-precision, values."""
    def __init__(self):
        super().__init__()
        self.rows, self.row, self.cell = [], [], None
    def handle_starttag(self, tag, attrs):
        if tag == "tr":
            self.row = []
        if tag in ["th","td"]:
            self.cell = ""
    def handle_data(self, data):
        if self.cell is not None:
            self.cell += data
    def handle_endtag(self, tag):
        if tag in ["th","td"] and self.cell is not None:
            self.row.append(self.cell.strip())
            self.cell = None
        if tag == "tr" and self.row:
            self.rows.append(self.row)


def read_table(folder, name, required=True):
    path = folder / f"{name}.csv"
    if not path.is_file():
        if required:
            raise FileNotFoundError(path)
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        if required:
            raise
        return pd.DataFrame()


def saved_moran(folder, manifest):
    # The complete original 08 screen includes the 2023Q3 peak missed by 09's two-quarter screen.
    if manifest["pipeline_version"] == "09-mvp-3":
        notebooks = sorted((folder.parent / "notebooks").glob("08*.ipynb"))
        if notebooks:
            notebook = json.loads(notebooks[0].read_text(encoding="utf-8"))
            for cell in notebook["cells"]:
                if cell.get("id") != "nb2-check-4-4":
                    continue
                for output in cell.get("outputs", []):
                    html = "".join(output.get("data",{}).get("text/html",[]))
                    parser = DisplayTable()
                    parser.feed(html)
                    if parser.rows and "p_Holm" in parser.rows[0]:
                        table = pd.DataFrame(parser.rows[1:], columns=parser.rows[0]).drop(columns="")
                        table = table.rename(columns={"p_Holm":"p_holm","Inseln":"islands"})
                        for name in ["I","p_perm","p_holm","islands"]:
                            table[name] = pd.to_numeric(table[name], errors="raise")
                        table["source"] = "saved 08; 999 permutations; Holm over 40 quarters; displayed precision"
                        return table
    table = read_table(folder, "founding_moran_screen")
    table["source"] = "09 model bundle; adjustment applies to its selected periods"
    return table


samples, count_rows, cox_rows, coefficients, comparisons = [], [], [], [], []
h3_rows, diagnostic_rows, moran_frames, sensitivity_frames = [], [], [], []
manifests, tables, sources = {}, {}, {}
for resolution, folder in sorted(RESULT_BUNDLES.items()):
    manifest_path = folder / "model_run_manifest.json"
    m = json.loads(manifest_path.read_text(encoding="utf-8"))
    if m["status"] != "completed" or m["resolution_m"] != resolution:
        raise ValueError(f"Select a completed {resolution} m bundle.")
    manifests[resolution] = m
    sources[str(resolution)] = dict(run_id=m["run_id"], pipeline=m["pipeline_version"],
        manifest_sha256=hashlib.sha256(manifest_path.read_bytes()).hexdigest())
    sample = read_table(folder, "sample_description")
    count_sample = sample.loc[(sample.model=="founding") & (sample.stage=="model_sample")].iloc[0]
    survival_sample = sample.loc[(sample.model=="survival") & (sample.stage=="model_sample")].iloc[0]
    samples.append(dict(resolution_m=resolution, cell_quarters=int(count_sample.rows),
        cells=int(count_sample.units), births=int(count_sample.events), zero_share=count_sample.zero_share,
        excluded_cell_quarters=int(count_sample.excluded), locations=int(survival_sample.units),
        site_quarters=int(survival_sample.rows), exits=int(survival_sample.events),
        right_censored=int(survival_sample.units-survival_sample.events),
        excluded_locations=int(survival_sample.excluded)))
    count = read_table(folder, "founding_nb2_hypotheses")
    count = count.loc[count.hypothesis_id.isin(["H1","H2a","H4_routes","H4_no_stop","H5"])].copy()
    count["resolution_m"] = resolution
    count_rows.append(count)
    cox = read_table(folder, "survival_cox_hypotheses").copy()
    cox["resolution_m"] = resolution
    cox_rows.append(cox)
    for model, filename in [("NB2","founding_nb2_results"),("Cox","survival_cox_results")]:
        table = read_table(folder, filename)
        table = table.loc[~table.term.str.startswith("period_")].assign(model=model, resolution_m=resolution)
        coefficients.append(table)
        if model == "NB2":
            mass = table.loc[table.term.eq("log_pop_access_ring_0_15")].iloc[0]
            count_rows.append(pd.DataFrame([dict(hypothesis_id="joint_market", estimate=mass.coefficient,
                std_error=mass.std_error, ci_lower=mass.ci_lower, ci_upper=mass.ci_upper,
                p_value=mass.p_value, resolution_m=resolution, coverage="context, not H1")]))
    comparisons.append(read_table(folder, "model_comparison").assign(resolution_m=resolution))
    groups = read_table(folder, "h3_group_diagnostics")
    group_effects = read_table(folder, "h3_population_effects")
    numerical_verified = "newton_step_se" in groups and bool(groups.newton_step_se.le(.05).all())
    h3 = read_table(folder, "h3_hypotheses").copy()
    h3["resolution_m"] = resolution
    h3["numerically_verified"] = numerical_verified
    h3["use"] = "conditional group contrast" if numerical_verified else "rerun required; original sparse NB2 fits unverified"
    h3_rows.append(h3)
    ph = read_table(folder, "survival_schoenfeld_screen")
    moran = saved_moran(folder, m).assign(resolution_m=resolution)
    moran_frames.append(moran)
    diagnostic_rows.append(dict(resolution_m=resolution,
        spatial_periods=len(moran), spatial_flags=int(moran.p_holm.lt(.05).sum()),
        maximum_moran_I=float(moran.I.max()), maximum_moran_period=moran.loc[moran.I.idxmax(),"period"],
        ph_flags=int(ph.p_holm.lt(.05).sum()), ph_terms=len(ph),
        maximum_absolute_ph_rho=float(ph.spearman_rho.abs().max()),
        h3_numerically_verified=numerical_verified))
    tables[resolution] = dict(groups=group_effects, group_diagnostics=groups, ph=ph)
    for filename in ["founding_block_hypotheses","founding_stock_sensitivity",
                     "survival_block_hypotheses","survival_age_effects"]:
        extra = read_table(folder, filename, required=False)
        if not extra.empty:
            sensitivity_frames.append(extra.assign(resolution_m=resolution, sensitivity=filename))

sample_table = pd.DataFrame(samples)
count_effects = pd.concat(count_rows, ignore_index=True)
cox_effects = pd.concat(cox_rows, ignore_index=True)
for col in ["estimate","ci_lower","ci_upper"]:
    cox_effects["HR" if col=="estimate" else "HR_"+col] = np.exp(cox_effects[col])
coefficient_table = pd.concat(coefficients, ignore_index=True)
model_comparison = pd.concat(comparisons, ignore_index=True)
h3_review = pd.concat(h3_rows, ignore_index=True)
diagnostics = pd.DataFrame(diagnostic_rows)
moran_quarters = pd.concat(moran_frames, ignore_index=True)
sensitivities = pd.concat(sensitivity_frames, ignore_index=True) if sensitivity_frames else pd.DataFrame()
display(sample_table)
display(diagnostics)

## 2. Figures

In [ ]:
def save_figure(fig, name):
    for extension in ["png","svg"]:
        fig.savefig(OUTPUT_DIR / f"{name}.{extension}", dpi=600, bbox_inches="tight")
    plt.close(fig)


def forest(table, identifiers, labels, filename, hazard=False):
    fig, ax = plt.subplots(figsize=(6.5,3.7))
    sizes = sorted(table.resolution_m.unique())
    for i, resolution in enumerate(sizes):
        selected = table.loc[table.resolution_m.eq(resolution)].set_index("hypothesis_id").reindex(identifiers)
        centre = np.exp(selected.estimate) if hazard else selected.estimate
        lower = np.exp(selected.ci_lower) if hazard else selected.ci_lower
        upper = np.exp(selected.ci_upper) if hazard else selected.ci_upper
        y = np.arange(len(identifiers)) + (i-(len(sizes)-1)/2)*.18
        ax.errorbar(centre, y, xerr=[centre-lower,upper-centre], fmt=["o","s","D"][i],
                    color=["0.0","0.4","0.65"][i], capsize=2, markersize=4, label=f"{resolution} m")
    ax.axvline(1 if hazard else 0, color="0.4", linestyle="--", linewidth=.8)
    ax.set_yticks(np.arange(len(identifiers)), labels)
    ax.invert_yaxis()
    ax.set_xlabel("Hazard Ratio mit 95%-Konfidenzintervall" if hazard else "Koeffizient mit 95%-Konfidenzintervall")
    ax.legend(loc="lower center", bbox_to_anchor=(.5,1.01), ncol=3)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x,_:f"{x:.2f}".replace(".",",")))
    apa7_axes(ax)
    fig.tight_layout()
    save_figure(fig, filename)


forest(count_effects, ["H1","H2a","joint_market"],
       ["Erreichbare Bevölkerung\n(Firmenbestand konstant)",
        "Erreichbarer Firmenbestand\n(Bevölkerung konstant)",
        "Bevölkerung und Firmenbestand\ngemeinsame Änderung"],
       "abbildung_1_gruendungszusammenhaenge")
forest(cox_effects, ["H2b","H2c"],
       ["Branchengleicher Bestand","Branchenfremder Bestand"],
       "abbildung_2_cox_hazard_ratios", hazard=True)

fig, ax = plt.subplots(figsize=(6.5,3.6))
for i, (resolution, group) in enumerate(moran_quarters.groupby("resolution_m")):
    group = group.sort_values("period")
    time = pd.PeriodIndex(group.period, freq="Q")
    x = time.year + (time.quarter-1)/4
    ax.plot(x, group.I, color=["0.0","0.4","0.65"][i], linestyle=["-","--",":"][i],
            linewidth=1.1, label=f"{resolution} m")
ax.axhline(0, color="0.6", linewidth=.7)
ax.set(xlabel="Kalenderjahr", ylabel="Moran’s I der Pearson-Residuen")
ax.set_xlim(2016,2025.9)
ax.set_xticks([2016,2018,2020,2022,2024,2025])
ax.yaxis.set_major_formatter(FuncFormatter(lambda y,_:f"{y:.2f}".replace(".",",")))
ax.legend(ncol=3)
apa7_axes(ax)
fig.tight_layout()
save_figure(fig, "abbildung_3_raeumliche_residuen")

print("Figures saved; captions and limitations are in ergebnisse_entwurf.md.")

## 3. Export tables and results text

In [ ]:
exports = {
    "01_stichprobe": sample_table,
    "02_gruendungseffekte": count_effects[[c for c in ["resolution_m","hypothesis_id","estimate","std_error",
        "ci_lower","ci_upper","p_value","coverage"] if c in count_effects]],
    "03_cox_effekte": cox_effects[[c for c in ["resolution_m","hypothesis_id","estimate","std_error",
        "ci_lower","ci_upper","p_value","HR","HR_ci_lower","HR_ci_upper"] if c in cox_effects]],
    "04_modellvergleich": model_comparison,
    "05_diagnosen": diagnostics,
    "06_h3_pruefbedarf": h3_review,
    "07_moran_quartale": moran_quarters,
    "08_koeffizienten_anhang": coefficient_table,
}
if not sensitivities.empty:
    exports["09_sensitivitaeten"] = sensitivities
for name, table in exports.items():
    table.to_csv(OUTPUT_DIR / f"{name}.csv", index=False, encoding="utf-8-sig")
(OUTPUT_DIR / "result_sources.json").write_text(json.dumps(sources, indent=2), encoding="utf-8")


def german(value, decimals=3):
    return f"{value:,.{decimals}f}".replace(",","_").replace(".",",").replace("_",".")


def md_table(frame):
    header = "| " + " | ".join(map(str,frame.columns)) + " |"
    separator = "| " + " | ".join(["---"]*len(frame.columns)) + " |"
    rows = ["| " + " | ".join(map(str,row)) + " |" for row in frame.itertuples(index=False,name=None)]
    return "\n".join([header,separator,*rows])



overview = count_effects.loc[count_effects.hypothesis_id.isin(["H1","H2a","H5"])]
overview = overview.pivot(index="hypothesis_id", columns="resolution_m", values="estimate")
overview = overview.map(lambda x:german(x)).reset_index()
overview.columns = ["Kontrast", *[f"{x} m" for x in overview.columns[1:]]]
summary = "# Ergebnisübersicht\n\n" + md_table(overview)
summary += "\n\n## Modellprüfung\n\n" + md_table(diagnostics)
summary += ("\n\nDie Intervalle der ursprünglichen Läufe verwenden Zellcluster. "
            "Verbliebene räumliche Abhängigkeit und die Cox-PH-Screens sind bei der Interpretation zu beachten. "
            "H4 und der Spartenvergleich von H5 sind nur teilweise beziehungsweise nicht operationalisiert.\n")
if not h3_review.numerically_verified.all():
    summary += "\nH3: Die alten sparsamen NB2-Gruppenfits sind numerisch nicht ausreichend geprüft. Vor einer Hypothesenentscheidung neu schätzen.\n"
summary += "\n## Ausgewählte Läufe\n\n"
summary += "\n".join(f"- {size} m: {m['run_id']} ({m['pipeline_version']})" for size,m in manifests.items())
(OUTPUT_DIR / "ergebnisuebersicht.md").write_text(summary+"\n", encoding="utf-8")
print(f"{len(exports)} tables, figures and overview: {OUTPUT_DIR}")